Before version 3.0, Spark operated with static execution plans: once the optimizer defined the processing path, the plan was immutable regardless of the surprises the data held on disk. [Adaptive Query Execution (AQE)](https://www.databricks.com/blog/2020/05/29/adaptive-query-execution-speeding-up-spark-sql-at-runtime.html) fundamentally changes this paradigm, acting as Spark's central nervous system by using runtime statistics to dynamically adjust the execution plan. Instead of blindly following a theoretical model, Spark observes real-time metrics—such as bytes read and actual partition sizes—to "learn" during execution and reoptimize the query. This shifts the engine from a "guess-and-execute" approach to a reactive system that understands the real data it is handling at every stage.



One of the most immediate benefits of this dynamic approach is the automatic [coalescing of shuffle partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#coalescing-post-shuffle-partitions). While Spark typically defaults to 200 shuffle partitions, a dataset that only contains 15 distinct keys after filtering would normally result in 185 empty partitions, creating massive scheduling overhead. AQE introduces the **AQE Shuffle Read** step, which detects these small or empty fragments and merges them. The technical difference is overwhelming: in a scenario without AQE, you might see tasks lasting from 11 milliseconds to 7 seconds—a massive gap indicating critical inefficiency. With AQE, Spark standardizes the workload, making tasks range between 2 and 7 seconds, preventing the cluster from wasting CPU cycles managing insignificant tasks and ensuring cores don't sit idle.



When Spark detects that a partition in a [Sort Merge Join](https://www.youtube.com/watch?v=jiWCPJtDE2c) is significantly larger than others, it activates its [skewed join optimization](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join) capability. Instead of letting a single core suffer through a massive data key (like a Customer ID with millions of transactions), Spark dynamically splits the skewed partition into smaller fragments. To enable this "surgical" intervention, properties like `spark.sql.adaptive.enabled` and `spark.sql.adaptive.skewJoin.enabled` must be active, preventing the pipeline from stalling at the 75th percentile of execution while a single thread struggles. This addresses the inherent flaw in the [Shuffle anatomy](https://ydmarinb.github.io/data-engineering/Spark/Optimization-in-Apache-Spark--Shuffle-partitions/), where the formula $hash(key) \pmod{shuffle\_partitions}$ condemns a single executor to process every instance of an over-represented key.



However, the real "Game Changer" for eliminating skew remains the Broadcast Join. In a traditional Sort Merge Join, Spark is forced to partition data by the join key, which creates unbalanced partitions if the data is skewed. The Broadcast Join breaks this limitation by sending a complete copy of the small table to all executors, eliminating the need for a shuffle on the large table entirely. Since you are not forced to partition by the join key, you gain total flexibility: you can apply a `df.repartition(n)` to your giant table to force a perfectly [uniform distribution](https://en.wikipedia.org/wiki/Continuous_uniform_distribution), as the join will occur locally on each node regardless of the key structure. Transitioning to a broadcast-based join can reduce processing time to almost a third, simply by decoupling the execution from the underlying key distribution.